In [ ]:
import sys
sys.path.append('../../')

from pathlib import Path

import phys_ml.visualization.vertex_visualization as vertvis
import phys_ml.visualization.base as vis
from phys_ml.analysis.vertex import *
from phys_ml.evaluation import vertex as verteval
from phys_ml.load_data.vertex import *
from phys_ml.trainer.vertex import *
from phys_ml.util import slurm_generate

data_dir = Path('/gpfs/data/fs71925/shepp123/frgs_6d/')
base_path = '/gpfs/data/fs71925/shepp123/PhysML/saves/vertex_24x6/run_results/'
rmse_df = pd.read_csv(base_path + 'reconstruction_results.csv')
classification_df = pd.read_csv(base_path + 'classification_results.csv')

## contents

&emsp;0&emsp;on real space  
&emsp;1&emsp;trained on all vertices  
&emsp;&emsp;1-1&emsp;simple autoencoder  
&emsp;&emsp;1-2&emsp;contrastive autoencoder  
&emsp;2&emsp;trained on only 2 phases  
&emsp;&emsp;2-1&emsp;exclude SC-phase  
&emsp;&emsp;&emsp;2-1-1&emsp;simple autoencoder  
&emsp;&emsp;&emsp;2-1-2&emsp;contrastive autoencoder  
&emsp;&emsp;2-2&emsp;exclude AFM-phase  
&emsp;&emsp;&emsp;2-2-1&emsp;simple autoencoder  
&emsp;&emsp;&emsp;2-2-2&emsp;contrastive autoencoder  
&emsp;&emsp;2-3&emsp;exclude FM-phase  
&emsp;&emsp;&emsp;2-3-1&emsp;simple autoencoder  
&emsp;&emsp;&emsp;2-3-2&emsp;contrastive autoencoder  
&emsp;3&emsp;trained on only 1 phase  
&emsp;&emsp;3-1&emsp;SC-phase  
&emsp;&emsp;3-2&emsp;AFM-phase  
&emsp;&emsp;3-3&emsp;FM-phase  
&emsp;4&emsp;correlation  
&emsp;5&emsp;convergence  

## visualize vertex

In [ ]:
files_names = ['tp0.000000_mu0.000000', 'tp0.270000_mu1.080000', 'tp0.500000_mu2.000000']
vertices = [AutoEncoderVertex24x6Dataset.load_from_file(data_dir / f'{fn}.h5') for fn in files_names]
vertvis.plot_compare_slices(vertices)

## vertex statistics

In [ ]:
verteval.vertex_statistics(data_dir)

## correlation

In [ ]:
cor_mat = np.load(f'cor_mat_vertex24x6.npy')
vertvis.plot_correlation(cor_mat, "Vertex Correlation")

## compare autoencoder models

In [ ]:
verteval.plot_all_rmses(rmse_df, figsize=(6, 6))

In [ ]:
grouped_df = rmse_df.groupby(['run_id', 'ld', 's'])
for (run_id, ld, s), group in grouped_df:
    verteval.print_rmses(group.to_dict(), f'{run_id}_ld{ld}_s{s}')

## convergence

In [ ]:
from pathlib import Path

base_path = '/gpfs/data/fs71925/shepp123/PhysML/saves/vertex_24x6/run_results/'
folders = sorted([f.name for f in Path(base_path).iterdir() if f.is_dir()])
labels = {
    '1_1': '1-1 all vertices, simple AE',
    '1_2': '1-2 all vertices, contrastive AE',
    '2_1_1': '2-1-1 exclude SC, simple AE',
    '2_1_2': '2-1-2 exclude SC, contrastive AE',
    '2_2_1': '2-2-1 exclude AFM, simple AE',
    '2_2_2': '2-2-2 exclude AFM, contrastive AE',
    '2_3_1': '2-3-1 exclude FM, simple AE',
    '2_3_2': '2-3-2 exclude FM, contrastive AE',
    '3_1': '3-1 SC only, simple AE',
    '3_2': '3-2 AFM only, simple AE',
    '3_3': '3-3 FM only, simple AE',
}

filtered_folders = folders  # set this to plot only some runs
filtered_labels = []
for f in filtered_folders:
    fsplit = f.split('_')
    pref, suff = '_'.join(fsplit[:-1]), fsplit[-1]
    filtered_labels.append(f"{labels[pref]} ({suff})")
tensorboard_data = vis.get_tensorboard_data(base_path, filtered_folders, filtered_labels)

gb = tensorboard_data.groupby('run')
((gb['wall_time'].max() - gb['wall_time'].min()) / 3600).round(2)  # hours

In [ ]:
vis.plot_loss_progress(tensorboard_data.dropna(), labels, y_max=0.1, log=False)

## compare phase classifier models

In [ ]:
verteval.plot_classification_results(classification_df)

In [ ]:
labels = list(AutoEncoderVertexDataset.phase_borders.keys())
for i, row in classification_df.iterrows():
    verteval.print_conf_mat(row['conf_mat'], f"{row['run_id']}_ld{row['ld']}_s{row['s']}", labels)

## compare vertex reconstructions

In [ ]:
vertex_path_dict = {float(fp.stem[2:6]): fp for fp in data_dir.glob('*.h5')}

In [ ]:
runs_to_plot = ['2_1_1_ld16']
tp_to_plot = [0.20, 0.33]
vertices = [AutoEncoderVertex24x6Dataset.load_from_file(vertex_path_dict[tp]) for tp in tp_to_plot]
predictions = {run_id: [np.load(base_path / run_id / 'predictions' / vertex_path_dict[tp].stem + '.npy') for tp in tp_to_plot] 
               for run_id in runs_to_plot}

In [ ]:
i = 18
axis = 3
nrows, ncols = 1, 2
slice_at = (i, i, i, i)
other_ks = set(range(1,4)) - {(axis + 1) // 2}
params_str = ', '.join([f'$k_{{{k}_{c}}}={sl}$' for (k, c), sl 
                        in zip([(k, c) for k in other_ks for c in ['x', 'y']], slice_at)])

for run_id, preds in predictions.items():
    for tp, true, pred in zip(tp_to_plot, vertices, preds):
        fn = vertex_path_dict[tp].stem
        true_slice = vertvis.get_mat_slice(true, axis, slice_at)
        pred_slice = vertvis.get_mat_slice(pred, axis, slice_at)
        plot_data = {'true': true_slice, 'reconstruction': pred_slice}
        vertvis.plot_compare_grid(plot_data, nrows, ncols, axis, None, figsize=(6, 4), 
                                  title=f'Reconstruction of vertex({fn}) at ({params_str})', vmin=0.0, vmax=22.5)